## EU REED

#### 1) Import packages, background databases and set project

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
import bw2analyzer as bwa

import matplotlib.pyplot as plt
import d3graph
import polyviz

10:32:02+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = Config()

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [4]:
import import_db

#### 2) Import foreground database 

In [5]:
bonsai = bd.Database('bonsai_231231')
bonsai_bio = bd.Database('bonsai_231231 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [6]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

In [7]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260210.xlsx',)
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

Extracted 4 worksheets in 0.42 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [8]:
importer.statistics()

Graph statistics for `reed` importer:
32 graph nodes:
	None: 32
194 graph edges:
	technosphere: 160
	production: 32
	biosphere: 2
194 edges to the following databases:
	bonsai_231231: 110
	reed: 82
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):




(32, 194, 0, 0)

In [9]:
importer.write_database()

100%|██████████| 32/32 [00:00<00:00, 492.89it/s]


10:32:08+0100 [info     ] Vacuuming database            
Created database: reed


In [10]:
pd.DataFrame(importer.unlinked)

""


In [31]:
reed = bd.Database('reed')
bonsai = bd.Database('bonsai_231231')

In [12]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [13]:
from IPython.display import display
display(df_reed[['code', 'comment', 'location', 'unit', 'database']])

,code,comment,location,unit,database
0,"drilling, soraluce, hour",proxy for drilling. There are multiple differe...,GB,hour,reed
1,"Soraluce, average",This inventory is calculated based on informat...,GB,unit,reed
2,"flamecutting, lpg, hour",This process represents flamecutting. The data...,DE,hour,reed
3,Goimek piece,"Does not include drilling, painting, deburring...",ES,unit,reed
4,alkyd paint,None,DE,tonne,reed
5,sawing,energy consumption for sawing,DE,hour,reed
6,"electricity production, GB",proxy for electricity,GB,kilowatt hour,reed
7,"generic machining, soraluce, kilogram",to be used as proxy when machining process is ...,GB,kilogram,reed
8,"drilling, soraluce, kilogram",proxy for drilling. There are multiple differe...,GB,kilogram,reed
9,cast iron production,proxy for cast iron,CN,kilogram,reed


In [14]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

#### 3) Importing sensordata
###### This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [24]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [25]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [26]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
goimek  OF10001_G_BQC_S8CF2G 0.0                    8.154403     11.789498   
                             1.0                   24.300319     11.436939   
                             2.0                   10.289347     14.996619   
                             3.0                   14.673875     15.127706   
                             4.0                    0.930889     11.800000   
...                                                      ...           ...   
wwr     OF10004_G_TWB_7SZ421 2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0             28001.024411  13

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [27]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [28]:
df = sensordata.reset_index()

sensordata_reed = (
    df.groupby(["company", "event_id", "reed tool"], as_index=False)
      .agg(
          energy_consumption=("energy_consumption", "sum"),
          hours=("hours", "sum"),
      )
)

#recalculate Power_Active based on the summed energy consumption and hours for each group
sensordata_reed["Power_Active"] = sensordata_reed["energy_consumption"] / sensordata_reed["hours"]

sensordata_reed = (
    sensordata_reed
    .set_index(["company", "event_id", "reed tool"])
    .sort_index()
)

In [29]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)
print(missing_tool_descriptions)

   company  Tool_Number
0   goimek          0.0
1   goimek          2.0
2   goimek          6.0
3   goimek         30.0
4   goimek         31.0
..     ...          ...
70     wwr       3121.0
71     wwr       3124.0
72     wwr       3135.0
73     wwr       3270.0
74     wwr       3350.0

[75 rows x 2 columns]


In [30]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)
print(event_totals)

                              energy_consumption  Power_Active       hours  \
company event_id                                                             
goimek  OF10001_G_BQC_S8CF2G        2.275491e+02     13.661594   18.073333   
        OF10002_G_BQC_S8CF2G        1.935815e+02     13.649169   15.077500   
        OF10003_G_BQC_S8CF2G        2.040927e+02     13.668632   15.678333   
        OF10004_G_BQC_S8CF2G        1.755478e+02     13.997188   13.026944   
komatsu OF10001_G_SLG_E8AJ1C        4.003197e+04   4942.288054   31.014444   
        OF10001_G_TXG_REGRHM        1.424581e+05   6105.037617   30.933611   
        OF10002_G_SLG_E8AJ1C        1.088048e+05   4582.621415   64.132500   
        OF10002_G_TXG_REGRHM        1.910862e+05   5903.993065   64.143889   
        OF10003_G_SLG_E8AJ1C        1.122193e+05   5156.690932   29.227778   
        OF10003_G_TXG_REGRHM        1.556712e+05   6810.420996   29.212778   
        OF10004_G_SLG_E8AJ1C        8.183841e+04   5320.882015  

In [31]:
#Split final df by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[("wwr", "OF10004_G_TWB_7SZ421")]
)
print(tool_energy_by_company_and_event)

  company              event_id           reed tool  energy_consumption  \
0     wwr  OF10004_G_TWB_7SZ421            drilling        54469.954977   
1     wwr  OF10004_G_TWB_7SZ421  machining, generic       679216.721518   
2     wwr  OF10004_G_TWB_7SZ421             milling       462947.465990   

       hours  seconds  
0   4.410833    15879  
1  78.791389   283649  
2  41.757222   150326  


#### 4) Switching in sensordata

In [15]:
import bw2data as bd
import bw_processing as bwp
import bw2analyzer as bwa
import bw2calc as bc
import bw2io as bi
import pandas as pd
import numpy as np

In [34]:
machining = reed.get(name='generic machining, soraluce', unit='hour')

electricity = bonsai.get(name='electricity', location='GB', unit='TJ')

In [28]:
lca = machining.lca(method=config.lcia_gwp100)
lca.score

31.232247767455263

In [41]:
for f in machining.exchanges():
    print(f)

Exchange: 1 hour 'generic machining, soraluce' (hour, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 1.523427185872317e-05 TJ 'electricity' (TJ, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 0.00010287812840043525 tonne 'chemicals nec' (tonne, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 9.000489662676822e-07 MEUR 'collected and purified water, distribution services of water' (MEUR, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 34.472252448313384 m3 'compressed air' (m3, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 4.566210045662101e-06 unit 'Soraluce, average' (unit, GB, None) to 'generic machining, soraluce' (hour, GB, None)>
Exchange: 4.566210045662101e-06 tonne 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, GB, None) to 'generic machining, soraluce' (hour, GB, None)>


In [36]:
electricity_amount = abs(lca.technosphere_matrix[lca.product_dict[electricity.id],lca.activity_dict[machining.id]])
electricity_amount

1.5234271813824307e-05

In [37]:
cols = pd.Index(['original','swapped'])

In [38]:
rows = pd.MultiIndex.from_tuples([
                 (electricity.id,machining.id),
                  ],names=['product','activity'])
rows


MultiIndex([(239359684924796936, 282081744960483328)],
           names=['product', 'activity'])

In [39]:
values = np.array([[electricity_amount], [electricity_amount*5]]).T


In [40]:
scenarios = pd.DataFrame(values,index=rows,columns=cols)
scenarios

,,original,swapped
product,activity,,
239359684924796936,282081744960483328,0.000015,0.000076


In [49]:
dp = bwp.create_datapackage(sequential=True)
dp.add_dynamic_array(
    matrix='technosphere_matrix',
    indices_array=scenarios.index.values.astype(bwp.INDICES_DTYPE),
    interface=scenarios.values,
    flip_array=np.full(scenarios.index.shape,True))
fu, data_objs , _ = bd.prepare_lca_inputs({machining:1},method=config.lcia_gwp100)

In [51]:
lca_a = bc.LCA(demand=fu,method=config.lcia_gwp100,
               data_objs=data_objs + [dp],use_arrays=True)

In [53]:
lca_a.lci()
lca_a.lcia()
#lca_a.keep_first_iteration()

In [58]:
results = []
for i in range(2):
    
    print(i)
    print(lca_a.score)
    results.append(lca_a.score)
    steel_cons = (lca_a.technosphere_matrix[lca_a.product_dict[electricity.id],lca_a.activity_dict[machining.id]])
    next(lca_a)

0
34.70654926957755
-7.617135906912154e-05
1
31.232247767455263
-1.5234271813824307e-05


#### 5) Select processes to be analysed

In [ ]:
reed.search('welding')

In [ ]:
bonsai.search('basic iron and steel gb')

In [ ]:
bonsai_bio.search('carbon dioxide')

In [ ]:
check_activity_type = print(lpg["activity type"])

In [ ]:
machining_processes = {
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='meter'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='meter'),
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='hour'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='hour'),
    "bending": reed.get(name='bending', location="DE", unit='hour'),
    "sawing": reed.get(name='sawing', location="DE", unit='hour'),
    "welding, manual": reed.get(name='welding, manual', location="GB", unit='hour'),
    "welding, robotic": reed.get(name='welding, robotic', location="GB", unit='hour'),
    "turning": reed.get(name='turning', location="GB", unit='kilogram'),
    "painting": reed.get(name='painting', location="DE", unit='m2'),
    "Casting": reed.get(name='Casting', location="DE", unit='tonne'),
    "sandblasting": reed.get(name='sandblasting', location="DE", unit='m2'),
}

In [ ]:
soraluce_machining = {
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='kilogram'),
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='hour'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='kilogram'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='hour'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='kilogram'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='hour'),
}

In [ ]:
use_case_machines = {
    "use_case_komatsu": reed.get(name='Komatsu piece', location="GB", unit='unit'),
    "use_case_wwr": reed.get(name='WWR piece', location="DE", unit='unit'),
    "use_case_goimek": reed.get(name='Goimek piece', location="ES", unit='unit'),
}


In [ ]:
soraluce_machines = {
"soraluce_komatsu": reed.get(name='soraluce machine, Komatsu', location="GB", unit='unit'),
"soraluce_wwr": reed.get(name='soraluce machine, WWR', location="DE", unit='unit'),  
"soraluce_goimek": reed.get(name='soraluce machine, Goimek', location="ES", unit='unit'),
"Soraluce, average": reed.get(name='Soraluce, average', location="GB", unit='unit'),
}

In [ ]:
# export the LCI to Microsoft Excel in an overview format
bi.export.excel.write_lci_excel(reed.name, objs=[steel], dirpath=Path.cwd())

#### 4) Defining LCIA functions

In [ ]:
def lca_score(process_dict, method):
    results = []
    for name, process in process_dict.items():
        lca = process.lca(method)
        results.append({
            "Process": name,
            "Location": process.get('location'),
            "Functional Unit": process.get('unit'),
            "Climate Impact (kg CO2eq)": lca.score
        })

    return pd.DataFrame(results)

In [ ]:
def run_contribution_analysis(demand_object,
                              lcia_method,
                              max_level=1,
                              cutoff=0.01):
    return bwa.utils.recursive_calculation_to_object(
        demand_object,
        lcia_method=lcia_method,
        max_level=max_level,
        cutoff=cutoff,
        as_dataframe=True,
    )

In [ ]:
def plot_contribution(df,
                      title='Contribution Analysis',
                      xlabel='CO2 eq. emissions (kg)',
                      drop_parent=True):
    df_plot = df.copy()

    # Drop the parent (the demand itself)
    if drop_parent:
        # Heuristic: parent is usually the one with maximum score
        parent_score = df_plot['score'].max()
        df_plot = df_plot[df_plot['score'] < parent_score]

    # Sort descending by contribution
    df_plot = df_plot.sort_values(by='score', ascending=False)

    f, ax = plt.subplots(figsize=(5, 2))
    sns.barplot(y='name', x='score', data=df_plot, ax=ax)

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=''
    )
    plt.tight_layout()
    plt.show()

#### 5) Calculate and present results

In [ ]:
df_results = lca_score(machining_processes, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
flamecutting = reed.get(name = 'flamecutting, lpg', location = "DE", unit = 'hour')

In [ ]:
# Another option is to use the recursive calculation to an object, which returns a DataFrame
flamecutting_ca = bwa.utils.recursive_calculation_to_object(flamecutting,
                                          lcia_method=config.lcia_gwp100,
                                          max_level=3,
                                          cutoff=0.001,
                                          as_dataframe=True,
                                          )
flamecutting_ca

In [ ]:
df_results = lca_score(soraluce_machining, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_results = lca_score(use_case_machines, config.lcia_gwp100)
display(df_results)

In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

#### 6) Run contribution analysis

In [ ]:
processes_results_ca = {}

for key, demand in soraluce_machining.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    processes_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
use_case_results_ca = {}

for key, demand in use_case_machines.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    use_case_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )